# 🕵️ Data Detective — Data Wrangling en Data Science
## ¿Podemos confiar en estos datos?

Somos Data Analysts / Data Scientists junior. RRHH nos entrega postulaciones para analizar perfiles, salarios y experiencia, pero los datos vienen "como vienen".

### 🎯 Misión
**🔎 Detectar → 🧹 Transformar → ✅ Validar → 📊 Analizar**

> No transformamos datos porque sí. Los transformamos para que sean útiles, consistentes y confiables para responder preguntas.

### Objetivos
- inspeccionar antes de analizar;
- detectar nulos, duplicados, tipos incorrectos y valores sospechosos;
- transformar números, textos y fechas;
- estandarizar categorías;
- validar transformaciones;
- entender por qué la calidad importa en Data Science.


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

df = pd.read_csv("/content/candidatos_data_detective.csv")
df.head()


## 🚨 Misión 1 — No toques nada todavía
Antes de limpiar, conozcamos el dataset.

**¿Qué representa una fila? ¿Cuántas filas tenemos? ¿Qué columnas existen? ¿Qué tipos aparecen?**

🧠 **Regla de oro: primero inspeccionar, después transformar.**


In [ ]:
print("Filas y columnas:", df.shape)
display(df.head())
df.info()


## 🕵️ Misión 2 — ¿Qué problemas detectamos?
Buscamos:
- valores faltantes;
- tipos sospechosos;
- categorías inconsistentes;
- valores imposibles;
- duplicados.

💬 **Pregunta para el chat:** ¿Cuál te parece más peligroso para un análisis y por qué?


In [ ]:
print("Valores faltantes:")
display(df.isna().sum().sort_values(ascending=False).head(10))
print("Duplicados exactos:", df.duplicated().sum())

print("\nCiudades:")
display(df["ciudad"].value_counts(dropna=False))
print("\nSeniorities:")
display(df["seniority"].value_counts(dropna=False))


## 🔢 Misión 3 — Los tipos importan
Un dato puede **verse como número y seguir siendo texto**.

Antes de ejecutar, ¿qué creés que pasará si intentamos calcular el salario promedio?


In [ ]:
print(df.dtypes)
try:
    print(df["salario_esperado"].mean())
except Exception as e:
    print("🚨 No podemos calcularlo directamente.")
    print(type(e).__name__, "-", e)


## 🧹 Misión 4 — Transformar salarios
Queremos convertir formatos como `$1.500.000`, `$ 1.500.000` o `1500000` en un número.

**Buena práctica:** conservamos una copia del dato original para poder auditar qué hicimos.


In [ ]:
df_limpio=df.copy()
df_limpio["salario_original"]=df_limpio["salario_esperado"]
df_limpio["salario_esperado"]=(df_limpio["salario_esperado"].astype("string")
    .str.replace("$","",regex=False).str.replace(" ","",regex=False)
    .str.replace(".","",regex=False))
df_limpio["salario_esperado"]=pd.to_numeric(df_limpio["salario_esperado"],errors="coerce")
display(df_limpio[["salario_original","salario_esperado"]].head(15))


## 🔎 Validamos
Una transformación no termina cuando ejecutamos el código.

**¿Ahora es numérico? ¿Cuántos nulos tenemos? ¿Hay valores absurdos?**


In [ ]:
print("Tipo:",df_limpio["salario_esperado"].dtype)
print("Nulos:",df_limpio["salario_esperado"].isna().sum())
display(df_limpio["salario_esperado"].describe())


## 🏙️ Misión 5 — Estandarizar categorías
`Córdoba`, `CORDOBA`, `cordoba`, `cba` pueden representar lo mismo para el negocio, pero Pandas los ve como categorías distintas.

**Limpiar también es hacer comparables distintas formas de escribir un mismo concepto.**


In [ ]:
df_limpio["ciudad"]=(df_limpio["ciudad"].astype("string").str.strip().str.lower())
map_ciudades={"cordoba":"Córdoba","cba":"Córdoba","buenos aires":"Buenos Aires","bs as":"Buenos Aires",
"rosario":"Rosario","mza":"Mendoza","mendoza":"Mendoza","salta":"Salta"}
df_limpio["ciudad"]=df_limpio["ciudad"].replace(map_ciudades)
display(df_limpio["ciudad"].value_counts(dropna=False))


## 👩‍💻 Misión 6 — Seniority
¿Qué problemas genera tener `Junior`, `junior`, `JR` y `Jr`?

Primero observamos; después normalizamos.


In [ ]:
print(df_limpio["seniority"].value_counts(dropna=False))
df_limpio["seniority"]=(df_limpio["seniority"].astype("string").str.strip().str.lower())
map_s={"jr":"Junior","junior":"Junior","ssr":"Semi Senior","semi senior":"Semi Senior",
"semi-senior":"Semi Senior","semisenior":"Semi Senior","sr":"Senior","senior":"Senior",
"lead":"Lead","líder":"Lead"}
df_limpio["seniority"]=df_limpio["seniority"].replace(map_s)
display(df_limpio["seniority"].value_counts(dropna=False))


## 🚨 Misión 7 — Missing ≠ Invalid

**Missing:** no tenemos el dato (`NaN`).

**Invalid:** tenemos un valor, pero no parece válido (`150 años`, `-3 años de experiencia`, score `999`).

> No necesariamente se resuelven igual.


In [ ]:
for col in ["edad","experiencia_anios","score_tecnico"]:
    df_limpio[col]=pd.to_numeric(df_limpio[col],errors="coerce")

print("Edades sospechosas")
display(df_limpio.loc[(df_limpio.edad<18)|(df_limpio.edad>70),["id_candidato","edad"]])
print("Experiencia sospechosa")
display(df_limpio.loc[(df_limpio.experiencia_anios<0)|(df_limpio.experiencia_anios>50),["id_candidato","experiencia_anios"]])
print("Scores sospechosos")
display(df_limpio.loc[(df_limpio.score_tecnico<0)|(df_limpio.score_tecnico>100),["id_candidato","score_tecnico"]])


In [ ]:
# Para esta práctica, marcamos valores fuera de rango como faltantes.
df_limpio.loc[~df_limpio.edad.between(18,70),"edad"]=np.nan
df_limpio.loc[~df_limpio.experiencia_anios.between(0,50),"experiencia_anios"]=np.nan
df_limpio.loc[~df_limpio.score_tecnico.between(0,100),"score_tecnico"]=np.nan
display(df_limpio[["edad","experiencia_anios","score_tecnico"]].describe())


## 📅 Misión 8 — Fechas
Las fechas pueden venir en formatos diferentes. Queremos una representación que Pandas pueda interpretar y luego usar para crear variables útiles.


In [ ]:
df_limpio["fecha_postulacion"]=pd.to_datetime(df_limpio["fecha_postulacion"],errors="coerce",dayfirst=True)
df_limpio["mes_postulacion"]=df_limpio["fecha_postulacion"].dt.to_period("M").astype("string")
df_limpio["dia_semana"]=df_limpio["fecha_postulacion"].dt.day_name()
display(df_limpio[["fecha_postulacion","mes_postulacion","dia_semana"]].head())


## 🔍 Misión 9 — Duplicados
Hay dos preguntas:
1. ¿Hay filas exactamente iguales?
2. ¿Hay candidatos repetidos según una clave de negocio?

> Un duplicado no siempre se detecta comparando toda la fila.


In [ ]:
print("Duplicados exactos:",df_limpio.duplicated().sum())
print("IDs duplicados:",df_limpio["id_candidato"].duplicated().sum())
display(df_limpio[df_limpio["id_candidato"].duplicated(keep=False)].sort_values("id_candidato"))


## 🧪 Misión 10 — Validación final
¿Cómo sabemos que nuestro dataset está mejor?

Revisamos:
- tipos;
- nulos;
- categorías;
- rangos;
- duplicados.

No buscamos "cero problemas" a cualquier costo. Buscamos **problemas conocidos y decisiones documentadas**.


In [ ]:
print("SHAPE:",df_limpio.shape)
print("\nNULOS"); display(df_limpio.isna().sum().sort_values(ascending=False).head(10))
print("\nTIPOS"); print(df_limpio.dtypes)
print("\nCIUDADES"); display(df_limpio.ciudad.value_counts(dropna=False))
print("\nDUPLICADOS EXACTOS:",df_limpio.duplicated().sum())


# 📊 Misión 11 — Ahora sí: Data Science
Limpiar no es el objetivo final. Ahora podemos responder preguntas.

- ¿Cuál es el salario esperado promedio por seniority?
- ¿Hay diferencias entre ciudades?
- ¿Cuál es el score técnico promedio?
- ¿Qué relación vemos entre experiencia y salario?

**Primero calidad. Después análisis.**


In [ ]:
resumen=(df_limpio.groupby("seniority",dropna=False)
    .agg(candidatos=("id_candidato","count"),
         salario_promedio=("salario_esperado","mean"),
         experiencia_promedio=("experiencia_anios","mean"),
         score_promedio=("score_tecnico","mean"))
    .sort_values("salario_promedio",ascending=False))
display(resumen)


In [ ]:
resumen_ciudad=(df_limpio.groupby("ciudad",dropna=False)
    .agg(candidatos=("id_candidato","count"),
         salario_promedio=("salario_esperado","mean"),
         score_promedio=("score_tecnico","mean"))
    .sort_values("candidatos",ascending=False))
display(resumen_ciudad)


# 💥 Desafío final — Code Review

Un compañero entrega:

```python
df["edad"] = df["edad"].fillna(0)
df["ciudad"] = df["ciudad"].str.lower()
df = df.dropna()
```

### 🚦 Clasificá
🟢 OK · 🟡 Depende · 🔴 Peligroso

Justificá al menos una decisión.

**Pista:** ¿qué problema intenta resolver? ¿qué información podría perder? ¿cómo comprobarías que funcionó?


# 🧠 El machete

```text
🔎 INSPECCIONAR
      ↓
🧹 TRANSFORMAR
      ↓
🚨 VALIDAR
      ↓
📊 ANALIZAR
      ↓
💡 COMUNICAR
```

> **No transformes datos sin saber por qué.  
> No des por buena una transformación sin validarla.**

### Data Science ≠ solamente modelar

La calidad de los datos condiciona todo lo que viene después.
